# Data anlysis of MQ dev results 


In [13]:
import os
import pandas as pd
import numpy as np
import re
from collections import Counter

import biotite.database.entrez as entrez
import biotite.sequence as seq
from biotite.sequence import ProteinSequence
import biotite.sequence.io.fasta as fasta

%load_ext autoreload
from protein_inference import * 
from extracting_sequences import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Importing the data

In [14]:
root_dir = '/data/MQ_with_FDR' 
# for Alphafold data
exposure = '/sites_exposure_w_FDR.txt"
f_path = "/data/HUMAN.fasta"

In [15]:
dfs = []

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if file == 'msms.txt':
            file_path = os.path.join(subdir, file)
            # Get the parent directory one level up
            parent_folder = os.path.basename(os.path.dirname(subdir))
            df = pd.read_csv(file_path, delimiter="\t")
            
            new_columns = {}
            for col in df.columns:
                if 'SUMO2' in col:
                    new_col_name = 'SUMO2' + col.split('SUMO2', 1)[1]
                    new_columns[col] = new_col_name
            
            df.rename(columns=new_columns, inplace=True)
            df['Dataset'] = parent_folder
            df['PSMId'] = df.apply(lambda x: f"{'decoy' if pd.notna(x['Reverse']) else 'target'}_0_{x['Scan number']}_{x['Raw file']}_{x['Charge']}_1", axis=1)

            dfs.append(df)

big_df = pd.concat(dfs, ignore_index=True)

/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/1711247926.py:9: DtypeWarning: Columns (55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/1711247926.py:9: DtypeWarning: Columns (55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/1711247926.py:9: DtypeWarning: Columns (55,56,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/1711247926.py:9: DtypeWarning: Columns (55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")


In [16]:
dfs = []

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if 'SUMO2_3-DVFQQQTGGSites.txt' in file:
            file_path = os.path.join(subdir, file)
            # Get the parent directory one level up
            parent_folder = os.path.basename(os.path.dirname(subdir))
            df = pd.read_csv(file_path, delimiter="\t")
            
            new_columns = {}
            for col in df.columns:
                if 'SUMO2' in col:
                    new_col_name = 'SUMO2' + col.split('SUMO2', 1)[1]
                    new_columns[col] = new_col_name
            
            df.rename(columns=new_columns, inplace=True)
            df['Dataset'] = parent_folder
            #df['PSMId'] = df.apply(lambda x: f"{'decoy' if pd.notna(x['Reverse']) else 'target'}_0_{x['Scan number']}_{x['Raw file']}_{x['Charge']}_1", axis=1)

            dfs.append(df)

sites_df = pd.concat(dfs, ignore_index=True)

/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/1017989954.py:9: DtypeWarning: Columns (30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")


In [17]:
sites_df['MS/MS ID'] = sites_df['MS/MS IDs'].str.split(';')
sites_df_long = sites_df.explode('MS/MS ID').reset_index(drop=True)
sites_df_long = sites_df_long.dropna(subset=['MS/MS ID'])
sites_df_long['MS/MS ID'] = sites_df_long['MS/MS ID'].astype(int)

duplicates_sites = sites_df_long[sites_df_long.duplicated(subset=['MS/MS ID', 'Dataset'], keep=False)]

# Display the duplicated rows
#duplicates_sites[['Proteins', 'Positions within proteins',  'Localization prob']]

sites_df_long_cleaned = sites_df_long.loc[sites_df_long.groupby(['MS/MS ID', 'Dataset'])['Localization prob'].idxmax()]

# multiple sites per msms text, could be reason for addition of data here. 

print(len(big_df))
big_df = big_df.merge(sites_df_long_cleaned, left_on=['id', 'Dataset'], right_on=['MS/MS ID', 'Dataset'], how='left', suffixes=('', '_sites_table'))
print(len(big_df))

1781257
1781257


In [18]:
dfs = []

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if file == 'evidence.txt':
            file_path = os.path.join(subdir, file)
            parent_folder = os.path.basename(os.path.dirname(subdir))
            df = pd.read_csv(file_path, delimiter="\t")
            
            new_columns = {}
            for col in df.columns:
                if 'SUMO2' in col:
                    new_col_name = 'SUMO2' + col.split('SUMO2', 1)[1]
                    new_columns[col] = new_col_name
            
            df.rename(columns=new_columns, inplace=True)
            df['Dataset'] = parent_folder
            #df['PSMId'] = df.apply(lambda x: f"{'decoy' if pd.notna(x['Reverse']) else 'target'}_0_{x['Scan number']}_{x['Raw file']}_{x['Charge']}_1", axis=1)

            dfs.append(df)

evi_df = pd.concat(dfs, ignore_index=True)

/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/125041966.py:8: DtypeWarning: Columns (53,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/125041966.py:8: DtypeWarning: Columns (53,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/125041966.py:8: DtypeWarning: Columns (53,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")
/var/folders/xb/lqsp1s4j623fywqz4zpxf0v80000gn/T/ipykernel_65989/125041966.py:8: DtypeWarning: Columns (53,54) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, delimiter="\t")


In [19]:
print(len(evi_df))

#evi_df_cleaned = evi_df.loc[evi_df.groupby(['id', 'Dataset'])['Localization prob'].idxmax()]
#print(len(evi_df_cleaned))

1750920


In [20]:
print(len(evi_df))
merged_df = evi_df.merge(big_df, left_on=['id', 'Dataset'], right_on=['Evidence ID', 'Dataset'], how='left', suffixes=('', '_msms_table'))

print(len(merged_df))

1750920
1781257


In [21]:
print(len(big_df))
print(len(sites_df))
print(len(sites_df_long))
print(len(evi_df))


1781257
303770
1047799
1750920


In [22]:
big_df = big_df[big_df['Contaminant'] != '+']
big_df['Reverse'].fillna('-', inplace=True)
big_df['Modified sequence'] = big_df['Modified sequence'].str.replace(r'st_o_SUMO2/3-DVFQQQTGG|ST_2_SUMO2/3-DVFQQQTGG|SBM_4_o_SUMO2/3-DVFQQQTGG|SBM_5_o_SUMO2/3-DVFQQQTGG|SBM_3_o_SUMO2/3-DVFQQQTGG|SBM_2_o_SUMO2/3-DVFQQQTGG|SBM_8_o_SUMO2/3-DVFQQQTGG|SBM_0_o_SUMO2/3-DVFQQQTGG', 'su', regex=True)
big_df['Positions within proteins'] = big_df['Positions within proteins'].str.split(';')
big_df['Positions within proteins'] = big_df['Positions within proteins'].apply(
    lambda x: [int(pos) for pos in x if isinstance(x, list) and pos.strip().isdigit()] if isinstance(x, list) else []
)

In [23]:
# mergind af data with dataframe
with open(f_path, "r") as file:
    fasta_file = fasta.FastaFile.read(file)

sequence_dict = fasta.get_sequences(fasta_file)
sequence_dict = {header.split('|')[1]: seq for header, seq in sequence_dict.items()}

exposure_info = pd.read_csv(exposure, delimiter=";")
exposure_info['seq'] = exposure_info['sequence'].str.extract(r'"(.*?)"')

def convert_to_number_list(str_value):
    str_value = str_value.strip('[]').replace('\n', '') #.replace(' ', ',') #.replace('.', '') 
    return [float(x) for x in str_value.split(',') if x.strip() != '']  # Convert only non-empty parts

def convert_to_string_list(str_value):
    str_value = str_value.strip('[]').replace('\n', '') #.replace(' ', ',')
    return [x.strip().strip("'").strip('"') for x in str_value.split(',') if x.strip() != ''] 

exposure_info['plddt'] = exposure_info['plddt'].apply(convert_to_number_list)
exposure_info['sse'] = exposure_info['sse'].apply(convert_to_string_list)

big_df = pd.merge(big_df, exposure_info, left_on='Uniprot_cannonised', right_on='outer_key', how='left') 
big_df['AF_seq'] = big_df['seq'].fillna('').astype(str)
big_df['AF_seq'] = big_df['AF_seq'].apply(lambda x: [x])

/Users/hmt128/miniconda3/lib/python3.11/site-packages/biotite/sequence/io/fasta/convert.py:248: UserWarning: ProteinSequence objects do not support selenocysteine (U), occurrences were substituted by cysteine (C)
  warnings.warn(


In [25]:
# extract isoforms 
def extract_following_symbols(string, list_of_strings):
    if not isinstance(string, str): 
        return []
    
    if not isinstance(list_of_strings, list):  
        return []
    
    result = []
    for item in list_of_strings:
        if isinstance(item, str) and string in item:
            index = item.find(string)
            # Extract the two characters following the match
            following_symbols = item[index + len(string):index + len(string) + 2]
            if following_symbols.startswith('-'):
                result.append(following_symbols)
    return result

big_df['isoforms'] = big_df.apply(
    lambda row: extract_following_symbols(row['Uniprot_cannonised'], row['Proteins']), axis=1
)

def combine_string_and_list(string, list_of_strings):

    if not isinstance(string, str):  # Ensure the string is valid
        string = ""
    if not isinstance(list_of_strings, list):  # Ensure the list is valid
        return []
    
    combined = [string + item for item in list_of_strings]
    combined.insert(0, string)  
    return combined

big_df['uniprot_w_isoforms'] = big_df.apply(
    lambda row: combine_string_and_list(row['Uniprot_cannonised'], row['isoforms']), axis=1
)

big_df['isoform_seqs'] = big_df['uniprot_w_isoforms'].apply(
    lambda key_list: [str(sequence_dict.get(key, None)) for key in key_list]
)

def match_any_col2(list1, list2):
    match_results = [any(item1 in item2 for item1 in list1) for item2 in list2]
    matched_peptides = [item2 for item2, match in zip(list2, match_results) if match]

    return match_results, matched_peptides

# not 0 indexed
def find_k_positions(seq):
    return [i - 1 for i, letter in enumerate(str(seq)) if letter == 'K']

big_df['k_positions'] = big_df['AF_seq'].apply(find_k_positions)

In [26]:
# To calculate motif adherence fetch surrounding sequences 

# function that takes a dataframe
# check if any of the peptides in 'Peptide sequences' matches w the string in 'seq'
# if K is at the end of the 'peprtide sequences', make sure it's followed by a E or D in seq
# takes the positions of the K in the string match in seq 
# takes the ['SUMO2/3-DVFQQQTGG site positions'] # column with list of numbers to see if the numbers match , return the overlapping numbers 
# return the matched seuqences +-7 aroung the site positions, it's in the beginning or end add -- to account for the missing letters

def extract_sublist(lst, position, number, placeholder=None):
    start = position - number
    end = position + number + 1

    result = []
    for i in range(start, end):
        if 0 <= i < len(lst): 
            result.append(lst[i])
        else: 
            result.append("-")
    return result

def extract_matching_sequences(df, seq_len_tp_extract, seqs_to_use, results_column, pos_column = 'Positions within proteins', res_pdllt_column = 'extracted_pddlt',  AF = False):

    def extract_surrounding_sequence(seq, position, seq_len_tp_extract):
        """Helper function to extract the sequence with padding around the position."""
        start = max(0, position - seq_len_tp_extract)
        end = min(len(seq), position + seq_len_tp_extract + 1)
        substring = seq[start:end]
    
        # Pad with '-' if we're at the beginning or end
        pad_before = max(0, seq_len_tp_extract - position)
        pad_after = max(0, seq_len_tp_extract - (len(seq) - position - 1))
        padded_substring = '-' * pad_before + substring + '-' * pad_after

        return padded_substring

    def process_row(row, seq_len_tp_extract, seqs_to_use, AF):
        """Processes a single row to extract sequences and match peptides."""
        peptide = row['Sequence']
        #print(peptide)
        #print(row[pos_column])
        
        site_positions = row[pos_column]
        seqs = row[seqs_to_use]
        row_sse = row['sse']
        row_pdllt = row['plddt']
        pos_peptide = row['Position in peptide']

        matched_sequences = []
        sses = []
        pdllts = []

        for site_pos in site_positions:
            #print(site_pos)
            for seq in seqs:
                match_found = False  
                surrounding_sequence = extract_surrounding_sequence(seq, site_pos, seq_len_tp_extract)
                middle_index = seq_len_tp_extract - 1
                
                if surrounding_sequence[middle_index] == 'K':
                    #print(surrounding_sequence[middle_index])
                    #print(surrounding_sequence[middle_index + 1])
                    
                    if pos_column == 'Positions within proteins': 
                        if peptide.endswith('K') and pos_peptide == len(peptide):
                            if surrounding_sequence[middle_index + 1] == 'D' or surrounding_sequence[middle_index + 1] == 'E':
                                if peptide in surrounding_sequence:
                                    #print('------- MATCH ---------')
                                    matched_sequences.append(surrounding_sequence)
                                    match_found = True
                                    #break  
                        else:   
                            if peptide in surrounding_sequence:
                                #print('------- MATCH ---------')
                                matched_sequences.append(surrounding_sequence)
                                match_found = True
                                #break 
                                
                        if AF and match_found:
                            sse = extract_sublist(row_sse, site_pos, seq_len_tp_extract)
                            sses.append(sse)
                            pdllt = extract_sublist(row_pdllt, site_pos, seq_len_tp_extract)
                            #print(pdllt)
                            pdllts.append(pdllt)
                            
                    elif pos_column == 'k_positions': 
                        matched_sequences.append(surrounding_sequence)
                        pdllt = extract_sublist(row_pdllt, site_pos, seq_len_tp_extract)
                        #print(pdllt)
                        pdllts.append(pdllt)
                        match_found = False
                        
                if match_found: 
                    break
        
        #print(matched_sequences)
        return matched_sequences, sses, pdllts 

    df[[results_column, 'extracted_sse', res_pdllt_column]] = df.apply(
    lambda row: pd.Series(process_row(row, seq_len_tp_extract, seqs_to_use, AF)), axis=1
)
    return df

max_length = big_df['Length'].max() + 1


In [28]:
big_df = extract_matching_sequences(big_df, max_length, 'isoform_seqs', 'matched_sequences') 

In [29]:
big_df = extract_matching_sequences(big_df, max_length, 'AF_seq', 'all_matched_sequences_AF', 'k_positions', 'all_surr_pddlt', AF = True) 

In [30]:
big_df = extract_matching_sequences(big_df, max_length, 'AF_seq', 'matched_sequences_AF', AF = True) 

In [31]:
big_df.to_csv(root_dir + '/output.tsv', sep='\t', index=False)
